# Omni NCM — 10k-sample run (does the train/val gap come from too little data?)

**The question this notebook answers.** Your strict run on ~4k unique samples ended at
train CE **0.144** (perplexity 1.15) while held-out text-L5 accuracy was **0.467**. Those cannot
both be honest unless the model memorised the 4k. So: produce **10,000 unique samples**, train the
identical 300 steps, and see whether held-out accuracy moves.

**Numbers to beat** (the 4k strict run, Colab):

| metric | 4k run |
|---|---:|
| train CE (last step) | 0.144 |
| val text_L5 | 0.4668 |
| val image_L5 | 0.5827 |
| conditional text_L5 (strict) | 0.523 |
| conditional text_L5 (AR) | 0.323 |
| constant floor, text_L5 | 0.183 |

**Runtime.** The producer is CPU- and network-bound, not GPU-bound.

| runtime | vCPU | 10k samples (1 worker) | credits |
|---|---:|---|---|
| G4 (T4) | 2 | ~2.5–3.5 h | cheapest per hour, long wall clock |
| A100 | 12 | ~1–1.5 h | ~3x faster, burns units faster |

Training is 300 steps twice and takes minutes on either. **Pick A100 if you want it inside an
hour**; G4 if you want to spend units slowly. Nothing here needs 80 GB of VRAM — the model is
10.44M params.

Run cells 1 → 7 top to bottom. Cell 4 (produce) is the long one; it prints an ETA every 60 s.


In [ ]:
import os, sys, subprocess

REPO = 'adi274903/ncm-omni'          # private
PKG  = os.environ.get('NCM_PKG', '/content/ncm_omni')

def _token():
    """Read-only clone access. Colab secret first, then env, then an authenticated gh."""
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t.strip()
    except Exception:
        pass
    t = os.environ.get('GITHUB_TOKEN')
    if t:
        return t.strip()
    try:
        return subprocess.check_output(['gh', 'auth', 'token'], text=True).strip()
    except Exception:
        return None

def _askpass(tok):
    """Hand the token to git through GIT_ASKPASS, never through argv or .git/config.

    The old cell pulled with the plain https URL, but the clone deliberately stripped the
    credential out of origin - so a re-run died with `could not read Username`. The token has to
    be re-supplied on every pull, and it must not land in the notebook output or on disk.
    """
    d = os.path.dirname(os.path.abspath(PKG))
    secret = os.path.join(d, '.git-tok')
    with open(secret, 'w') as f:
        f.write(tok)
    os.chmod(secret, 0o600)
    p = os.path.join(d, '.git-askpass.sh')
    with open(p, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in\n  *sername*) echo x-access-token ;;\n'
                '  *) cat "{}" ;;\nesac\n'.format(secret))
    os.chmod(p, 0o700)
    return p, secret

def _git(args, tok=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    junk = []
    if tok:
        ap, secret = _askpass(tok)
        junk = [ap, secret]
        env.update(GIT_ASKPASS=ap, GIT_ASKPASS_REQUIRE='force')
    try:
        # credential.helper= clears any stale helper so askpass is the only source
        return subprocess.check_output(['git', '-c', 'credential.helper='] + args,
                                       text=True, stderr=subprocess.STDOUT, env=env).strip()
    finally:
        for f in junk:
            if os.path.exists(f):
                os.remove(f)

assert PKG.startswith('/content') or os.environ.get('NCM_PKG'), 'refusing to touch ' + PKG
tok = _token()
url = 'https://github.com/%s.git' % REPO
if os.path.isdir(os.path.join(PKG, '.git')):
    try:
        # Fetch + hard reset, not pull: the checkout is a --depth 1 graft, so a rewound history
        # cannot fast-forward, and run/cfg is rewritten by every invocation. The repo holds no
        # edits worth keeping (run/ is gitignored), so landing exactly on origin/main is right.
        _git(['-C', PKG, 'fetch', '--depth', '1', 'origin', 'main'], tok)
        _git(['-C', PKG, 'reset', '--hard', 'FETCH_HEAD'], tok)
    except subprocess.CalledProcessError as e:
        # Re-clone, but keep run/ (shards + checkpoints). A plain `rm -rf` here silently deletes
        # hours of training because run/ is gitignored and therefore not in the repo.
        print('update failed -> re-cloning (keeping run/):',
              str(e.output).strip().splitlines()[-1])
        import shutil
        tmp = PKG.rstrip('/') + '.new'
        shutil.rmtree(tmp, ignore_errors=True)
        _git(['clone', '--depth', '1', url, tmp], tok)
        keep = os.path.join(PKG, 'run')
        if os.path.isdir(keep):
            shutil.rmtree(os.path.join(tmp, 'run'), ignore_errors=True)
            shutil.move(keep, os.path.join(tmp, 'run'))
        shutil.rmtree(PKG, ignore_errors=True)
        os.rename(tmp, PKG)
else:
    _git(['clone', '--depth', '1', url, PKG], tok)

assert os.path.exists(os.path.join(PKG, 'core.py')), 'clone landed without core.py'
os.chdir(PKG)
if os.environ.get('NCM_SKIP_PIP') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                           'datasets', 'soundfile', 'av'])

cfg = os.path.join(PKG, '.git', 'config')
assert 'x-access-token' not in open(cfg).read(), 'credential leaked into .git/config'

import torch
print('repo   ', REPO, '| commit', subprocess.check_output(
    ['git', '-C', PKG, 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('torch  ', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('workdir', os.getcwd())


In [ ]:
# HF token + hardware probe. Unauthenticated Hub requests answer 429 on bursts of stream opens,
# which silently drops sources and changes the mix. The token is read from Colab secrets and
# never printed.
import os, shutil, subprocess, sys

def _hf_token():
    try:
        from google.colab import userdata
        for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            try:
                t = userdata.get(k)
                if t:
                    return t.strip()
            except Exception:
                pass
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k):
            return os.environ[k].strip()
    return None

t = _hf_token()
if t:
    os.environ["HF_TOKEN"] = t           # picked up by huggingface_hub in-process and by children
    print("HF token: loaded (%d chars)" % len(t))
else:
    print("HF token: MISSING -> add a Colab secret named HF_TOKEN "
          "(key icon, left sidebar). Without it the producer will 429 and skip sources.")

try:
    from huggingface_hub import whoami
    print("HF auth: ", whoami()["name"] if os.environ.get("HF_TOKEN") else "anonymous")
except Exception as e:
    print("HF auth check skipped:", type(e).__name__)

cpu = os.cpu_count()
ram = shutil.disk_usage("/").total / 1e9
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
except Exception:
    gpu, vram = "unknown", 0
def _ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                return int(line.split()[1]) / 1e6
    except Exception:
        pass
    return 0.0
print(f"cpu {cpu} threads | RAM {_ram_gb():.0f} GB | disk {ram:.0f} GB | {gpu} ({vram} GB)")
print("WORKERS note: 1 is the only proven configuration. >1 sharded fine at 4 but hung at 8.")


In [ ]:
# Invariants: level k may not see level k+1, the mask is wired, every modality packs.
!python run_omni.py check 2>&1 | tail -12


In [ ]:
# ---- PRODUCE: 10,000 unique samples ------------------------------------------------
# What one sample is: one document / image+caption / audio clip / video clip, encoded into the
# 6-level pyramid for its modality and packed to 504 positions (base_len=64). 10k samples
# ~= 5.04M packed positions, written as shards of 64 plus the held-out blocks
# (32 text / 24 image / 16 audio / 8 video), which worker 0 writes first.
import json, os, subprocess, sys, time

BUDGET  = 10000   # unique samples. 8k = ~73 min on the laptop, 20k = ~3 h
WORKERS = 1       # >1 is EXPERIMENTAL: proven at 1, no gain at 4, hang at 8

os.makedirs("run/stream", exist_ok=True)
SHARD = 64
LOG = "run/produce.log"

def shards():
    d = "run/stream"
    return len([f for f in os.listdir(d) if f.endswith(".npz")]) if os.path.isdir(d) else 0

def done():
    return os.path.exists(os.path.join("run/stream", "DONE"))

if done() and shards() >= BUDGET // SHARD - 2:
    print("stream already complete:", shards(), "npz files - skipping production")
else:
    log = open(LOG, "a")
    p = subprocess.Popen([sys.executable, "-u", "run_omni.py", "produce",
                          "--budget", str(BUDGET), "--workers", str(WORKERS), "--foreground"],
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print(f"producer pid {p.pid} | budget {BUDGET} | workers {WORKERS} -> {LOG}")
    t0, last_print = time.time(), 0.0
    while True:
        time.sleep(20)
        n, el = shards(), time.time() - t0
        samples = n * SHARD
        if el - last_print >= 60 or done():
            rate = samples / el if el else 0
            eta = (BUDGET - samples) / rate / 60 if rate > 0.05 else float("nan")
            print(f"[{el/60:5.1f} min] {n:4d} shards | ~{samples:5d}/{BUDGET} samples | "
                  f"{rate:4.2f} samples/s | ETA {eta:5.0f} min", flush=True)
            last_print = el
        if done() or p.poll() is not None:
            break
    if not done():
        print("producer exited without DONE - tail of log:")
        print(subprocess.check_output(["tail", "-25", LOG], text=True))
    else:
        st = json.load(open("run/stream/stats.json"))
        print("DONE", {k: st.get(k) for k in ("samples", "shards", "elapsed_min", "per_source")})


In [ ]:
# ---- TRAIN on the new stream -------------------------------------------------------
# Both honest arms, from scratch: a stale checkpoint from the 4k run would make eval compare a
# model trained on old data against a new stream. block/ (legacy, leaky) is deleted so it can
# never appear in the table.
!rm -rf run/train_strict run/train_ar run/train_block
!python run_omni.py train --arm strict --steps 300 2>&1 | tee run/train_strict.log


In [ ]:
# AR arm on the same data: the comparison the claim depends on.
!python run_omni.py train --arm ar --steps 300 2>&1 | tee run/train_ar.log


In [ ]:
# ---- EVAL: conditional table (the quotable one) + free-running (structurally VOID) ----
!python run_omni.py eval 2>&1 | tee run/eval.log


In [ ]:
# ---- VERDICT: memorisation or too little data? ------------------------------------
import json, os, re

REF_4K = dict(train_ce=0.144, val_text_L5=0.4668, val_image_L5=0.5827,
              cond_text_L5=0.523, cond_ar_text_L5=0.323)

def tail(path, n=4000):
    return open(path).read()[-n:] if os.path.exists(path) else ""

def last_val(log):
    m = re.findall(r"val acc: (\{.*?\})", tail(log), re.S)
    return eval(m[-1]) if m else {}

def last_loss(log):
    m = re.findall(r'"loss": ([0-9.]+)', tail(log))
    return float(m[-1]) if m else float("nan")

vs, va = last_val("run/train_strict.log"), last_loss("run/train_strict.log")
tab = json.load(open("run/eval_table.json"))
cond = tab["conditional"]
floor = tab["baseline_by_level"].get("5", tab["baseline"])
st, ar = cond.get("level-parallel(strict)", {}), cond.get("autoregressive", {})
n_shards = len([f for f in os.listdir("run/stream") if f.startswith("shard_") and f.endswith(".npz")])
n_samples = n_shards * 64

print(f"unique samples in stream : ~{n_samples}")
print(f"strict train CE          : {va:.4f}   (4k run: {REF_4K['train_ce']:.3f})")
print(f"val text_L5              : {vs.get('text_L5', float('nan')):.4f}   (4k run: {REF_4K['val_text_L5']:.4f})")
print(f"val image_L5             : {vs.get('image_L5', float('nan')):.4f}   (4k run: {REF_4K['val_image_L5']:.4f})")
print(f"conditional text_L5      : strict {st.get('text_L5', float('nan')):.3f} | "
      f"AR {ar.get('text_L5', float('nan')):.3f} | constant {floor:.3f}")
gap = vs.get("text_L5", 0) - REF_4K["val_text_L5"]
print()
if va > 0.35 and gap > 0.03:
    print("VERDICT: the 0.144 was memorisation. With 2.5x the unique data the same 300 steps "
          "generalise better -> the 10M harness was data-starved, and a 315M run with 20x "
          "positions/param is worth the H100 spend.")
elif va < 0.35 and abs(gap) <= 0.03:
    print("VERDICT: val did not move while train CE stayed ~0.14. More data at this size is not "
          "the lever - the model is capacity-limited, so the next honest step is 85M params on "
          "the SAME data, not a bigger corpus.")
else:
    print("VERDICT: mixed. train CE %.3f, val text_L5 %+.3f vs the 4k run. Send both numbers; "
          "do not quote either table until the two arms agree in direction." % (va, gap))

dst = "/content/eval_table_10k.json"
try:
    open(dst, "w").write(json.dumps(tab, indent=2))
    print("\nsaved", dst)
except Exception as e:
    print("\nnot saved:", type(e).__name__)
try:
    os.makedirs("/content/drive/MyDrive/ncm", exist_ok=True)
    json.dump(tab, open("/content/drive/MyDrive/ncm/eval_table_10k.json", "w"), indent=2)
    print("also saved to Drive/ncm/")
except Exception:
    pass


## Reading the result

- **Only the conditional table is quotable.** Free-running accuracy starts with a 1-of-256 guess
  at level 0; eval labels any margin under 0.05 a tie, not a win.
- **The `level-parallel(LEGACY-leaky)` row must not exist** — cell 5 deletes `train_block`. If it
  appears, its text column is 1.000 at every level, which is the leak, not a result.
- **Per modality, never the mean.** Audio and video have collapsed codecs (~143 and ~22 distinct
  codes in their holdout), so they drag any mean down without informing it.
- **Text's level-5 jump is partly codec inversion** (L4 0.125 -> L5 0.477 on the 4k run): the
  coarsest levels nearly determine the finest. Compare strict vs AR *within* a level, not across.

## If the producer stalls

Cell 4 printing `~0 samples` for more than five minutes means the stream is throttled or a
sharded read is hanging: set `WORKERS = 1` (it already is) and check that the HF token cell said
`loaded`. Every worker logs to `run/producer.log`; `tail -20 run/producer.log` shows 429s and
per-source fallbacks.
